In [1]:
import pandas as pd

spotify = pd.read_csv("../data/processed/spotify_tweets.csv")

print(spotify.shape)
spotify.head()

(74618, 7)


,tweet_id,author_id,inbound,created_at,text,response_tweet_id,in_response_to_tweet_id
0,848,SpotifyCares,False,Tue Oct 31 22:28:16 +0000 2017,@115887 Hmm. Can you try restarting your devic...,849,850.0
1,849,115887,True,Tue Oct 31 23:36:20 +0000 2017,@SpotifyCares doesn’t work and i even tried de...,851,848.0
2,851,SpotifyCares,False,Tue Oct 31 23:39:03 +0000 2017,@115887 Could you send us a DM with your accou...,NaN,849.0
3,850,115887,True,Tue Oct 31 21:41:37 +0000 2017,@SpotifyCares Premium &amp; when i️ have it on...,848,852.0
4,852,SpotifyCares,False,Tue Oct 31 21:04:13 +0000 2017,"@115887 Thanks. Just to be sure, are you Free ...",850,853.0


In [2]:
customers = spotify[spotify["inbound"] == True].copy()
support = spotify[spotify["inbound"] == False].copy()

print("Customers:", len(customers))
print("Spotify replies:", len(support))

Customers: 31353
Spotify replies: 43265


In [4]:
# Convert both IDs to the same datatype
customers["response_tweet_id"] = pd.to_numeric(
    customers["response_tweet_id"],
    errors="coerce"
)

support["tweet_id"] = pd.to_numeric(
    support["tweet_id"],
    errors="coerce"
)

# Build customer → Spotify reply pairs
pairs = customers.merge(
    support,
    left_on="response_tweet_id",
    right_on="tweet_id",
    suffixes=("_customer", "_support")
)

pairs = pairs[["text_customer", "text_support"]]

print("Reply pairs:", len(pairs))
pairs.head()

Reply pairs: 24481


,text_customer,text_support
0,@SpotifyCares doesn’t work and i even tried de...,@115887 Could you send us a DM with your accou...
1,@SpotifyCares Premium &amp; when i️ have it on...,@115887 Hmm. Can you try restarting your devic...
2,@SpotifyCares iphone 7+ and i have the most re...,"@115887 Thanks. Just to be sure, are you Free ..."
3,"@SpotifyCares Yes, multiple times. No changes....",@115889 Sorry to hear that. The Spotify app on...
4,@SpotifyCares 2/2... and there is no way to ma...,@115889 Got it. It's not possible at the momen...


In [5]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

reply_vectorizer = TfidfVectorizer(
    stop_words="english",
    max_features=5000
)

reply_matrix = reply_vectorizer.fit_transform(
    pairs["text_customer"]
)

print(reply_matrix.shape)

(24481, 5000)


In [7]:
def generate_reply(user_message):

    user_vector = reply_vectorizer.transform([user_message])

    similarity = cosine_similarity(
        user_vector,
        reply_matrix
    )

    best_match = similarity.argmax()

    return pairs.iloc[best_match]["text_support"]

In [8]:
test = "I was charged twice for Premium"

print("Customer:")
print(test)

print("\nSpotify AI:")
print(generate_reply(test))

Customer:
I was charged twice for Premium

Spotify AI:
@671882 Hey Chino, that doesn't sound good! Can you DM us your account's email address? We'll take a look backstage /JN https://t.co/ldFdZRiNAt


In [9]:
generate_reply("My music keeps skipping")

"@615220 Hi there! That's not cool. Does logging out, restarting your device, and logging back into Spotify help? Keep us in the loop /JI"

In [10]:
generate_reply("I can't log into my account")

"@214781 Hey Andy! Can you tell us more about what's happening? We'll see what we can suggest /RH"

In [11]:
import joblib
import os

os.makedirs("../results", exist_ok=True)

joblib.dump(reply_vectorizer, "../results/reply_vectorizer.pkl")
joblib.dump(reply_matrix, "../results/reply_matrix.pkl")

pairs.to_csv("../results/reply_pairs.csv", index=False)

print("Reply engine saved successfully!")

Reply engine saved successfully!
